##### Install libraries:

In [36]:
!pip3 install tldextract
!pip3 install langdetect -q

!pip3 install google-search-results -q
!pip3 install --upgrade lxml_html_clean

!pip3 install openai==1.55.3 -q
!pip3 install --upgrade openai

!pip3 install html2text -q
!pip3 install zenrows -q

##### Keys for API

In [34]:
##### request keys from admin

##### Import libraries:

In [37]:
import os, warnings, re

from openai import OpenAI
import openai

import httpx, html2text

from zenrows import ZenRowsClient

import tldextract, json, requests
from concurrent.futures import ThreadPoolExecutor, as_completed

from datetime import datetime as dt, timedelta
from datetime import timedelta as td
from dateutil.relativedelta import relativedelta

from google.colab import output
from google.colab import files

from langdetect import detect
from nltk.corpus import stopwords

from serpapi import GoogleSearch
from tqdm import tqdm

import numpy as np
import pandas as pd

tqdm.pandas()

##### Search request:

In [13]:
request_en = "Risk management platform for FinTech, FinTech risk management solution, FinTech-focused risk control system, Platform for managing risks in FinTech, Risk mitigation platform tailored for FinTech"

##### Date frame (days):

In [5]:
frame = 183

##### Propmts:

In [66]:
role = "You are the FinTech expert"

web_search_prompt = f"Extract the cases of FinTech companies or commercial banks launching the technological product along with a mention\
 of a specific technology, product, business value and company, published in the last {frame} days on the topic {request_en}"

case_prompt = "Extract the cases of FinTech companies or commercial banks launching the technological product along with a mention\
 of a specific technology, product, business value of technology (or product) application and company from this text (no introduction,\
  not list, at least five sentences for each case, if no cases - output 'NO'): "

class_prompt = "Is case of the FinTech company or commercial bank launching\
 a technological product in {request_class} along with a mention of a specific technology mentioned in this text? Output only '1' if yes, or '0' if no: "

##### CDFs:

In [67]:
# create clients
def get_openai_client():
    return openai.Client(api_key=os.environ["OPENAI_API_KEY"], http_client=httpx.Client(proxy=OPENAI_PROXY))

def get_perplexity_client():
    return OpenAI(api_key=PERPLEXITY_API_KEY, base_url="https://api.perplexity.ai", http_client=httpx.Client(proxy=OPENAI_PROXY))

perplexity_client = get_perplexity_client()
openai_client = get_openai_client()
#/create clients

# generate urls
def web_search_urls(z):

  prompt = web_search_prompt.format(request_en=z)
  resp = openai_client.responses.create(model="gpt-4.1", tools=[{"type": "web_search"}], input=prompt)

  links = re.findall(r'https://\S+', resp.output_text)
  links = [re.sub(r'[),.;:\]\}>]+$', '', url) for url in links]
  links = list(set([url for url in links if "wikipedia.org" not in url]))

  return links

def perplexity_search_urls(z):

  prompt = web_search_prompt.format(request_en=z)
  messages = [{"role": "system", "content": "You are technology and business expert"}, {"role": "user", "content": prompt}]

  resp = perplexity_client.chat.completions.create(model="sonar-pro", messages=messages, temperature=0,)

  links = [re.sub(r'[),.;:\]\}>]+$', '', url) for url in resp.citations]
  links = list(set([url for url in links if "wikipedia.org" not in url]))

  return links
#/generate urls

# parse urls
def get_text_content(html_content: str) -> str:

    text_maker = html2text.HTML2Text()
    text_maker.ignore_links = True

    return text_maker.handle(html_content)

def parse_url(url: str, verbose: bool = False, time_out: int = 10) -> str:

    """
    Парсит URL с помощью ZenRows, используя ScraperAPI в качестве запасного варианта.
    Возвращает извлеченный текст или пустую строку в случае неудачи.
    """

    zenrows_key = ZENROWS_KEY
    scraperapi_key = SCRAPERAPI_KEY

    scraped_text = ''

    # --- Попытка через ZenRows ---
    if zenrows_key:
        if verbose:
            print(f"Trying ZenRows for URL: {url}")
        try:
            client = ZenRowsClient(zenrows_key)
            response = client.get(url, params={'premium_proxy': 'true'}, timeout=time_out)

            if response.status_code == 200:
                scraped_text = get_text_content(response.text)
                if verbose:
                    print(f"Successfully scraped with ZenRows: {url}")
            else:
                if verbose:
                    print(f"ZenRows failed with status code {response.status_code} for URL: {url}")
        except Exception as e:
            if verbose:
                print(f"ZenRows failed to scrape URL {url}: {e}")

    # --- Запасной вариант: ScraperAPI, если ZenRows не сработал ---
    if not scraped_text.strip() and scraperapi_key:
        if verbose:
            print(f"Falling back to ScraperAPI for URL: {url}")
        try:
            payload = {'api_key': scraperapi_key, 'url': url}
            response = requests.get('https://api.scraperapi.com/', params=payload, timeout=time_out)

            if response.status_code == 200:
                scraped_text = get_text_content(response.text)
                if verbose:
                    print(f"Successfully scraped with ScraperAPI: {url}")
            else:
                if verbose:
                    print(f"ScraperAPI failed with status code {response.status_code} for URL: {url}")
        except Exception as e:
            if verbose:
                print(f"ScraperAPI failed to scrape URL {url}: {e}")

    if not scraped_text.strip():
        print(f'Scraping failed for URL: {url}')
        return None

    return scraped_text
#/parse urls

# extract case
def case_extraction(text, case_prompt=case_prompt):

  client = OpenAI()

  system_prompt = role
  messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": case_prompt + "'" + text + "'"}]
  responce = client.chat.completions.create(model="gpt-4o-mini", messages=messages, temperature=0).choices[0].message.content

  return responce

def case_russian(text):

  client = OpenAI()

  system_prompt = role
  messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": "Translate this text into russian language (don't mention text language in the output): " + "'" + text + "'"}]
  responce = client.chat.completions.create(model="gpt-4o-mini", messages=messages, temperature=0).choices[0].message.content

  return responce
#/extract case

# content classification
def content_classification(text, class_prompt=class_prompt, counter=0, model="gpt-4o-mini", system_prompt = None):

    client = OpenAI()

    system_prompt = role + ".Return answer in json {'synergy': 'int'}"

    messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": f'task: {class_prompt} : "text: {text}" '}]
    response = client.chat.completions.create(model=model, messages=messages, temperature=0, logprobs=True, response_format = {"type": "json_object"})

    element = json.loads(response.choices[0].message.content)['synergy']
    token_list = [element.token for element in response.choices[0].logprobs.content]

    element_str = str(element)

    if element_str in token_list:

        synergy = bool(element)

        index = token_list.index(element_str)
        probs = np.exp(response.choices[0].logprobs.content[index].logprob)
        token = int(response.choices[0].logprobs.content[index].token)
    else:
        if counter < 5:
            counter +=1
            print(f'Try #{counter}\ntoken: {token}\nsynergy: {synergy}')
            return news_classification(text, class_prompt, counter=counter)
        else:
            print('попытки закончились')
            probs = None
            synergy = False

    if int(synergy)==True and probs > 0.95:
      synergy = 1
    else:
      synergy = 0

    return synergy
# content classification

# parallelization
def parallel_text_func(texts, func, max_workers: int = 10):
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        results = list(tqdm(executor.map(lambda x: func(x), texts), total=len(texts)))
    return results
# parallelization

##### Supress warnings:

In [10]:
warnings.filterwarnings('ignore')

##### Define time period:

In [12]:
date_start = (dt.strptime(dt.now().strftime("%m/%d/%Y"), "%m/%d/%Y") - timedelta(days=183)).strftime("%m/%d/%Y")
date_final = (dt.strptime(dt.now().strftime("%m/%d/%Y"), "%m/%d/%Y")).strftime("%m/%d/%Y")

##### Generate Web Search URLs:

In [ ]:
openai_urls = [web_search_urls(i) for i in tqdm(request_en.split(','))]
openai_urls = list(set([x for sub in openai_urls for x in sub]))

perplexity_urls = [perplexity_search_urls(i) for i in tqdm(request_en.split(','))]
perplexity_urls = list(set([x for sub in perplexity_urls for x in sub]))

case_urls = list(set(openai_urls + perplexity_urls))
df_web_urls = pd.DataFrame({'URL': case_urls})

print('\n\nNumber of case URLs:', len(case_urls))

100%|██████████| 5/5 [01:17<00:00, 15.47s/it]


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>



Number of case URLs: 27


##### Generate News Search URLs:

In [16]:
%%time

urls = []

for j in tqdm(list(request_en.split(','))):

  for i in tqdm(range(0, 100, 1)):

    params = {'api_key': SERPAPI_KEY, 'engine': 'google', 'tbm': 'nws', "tbs":("cdr:1,cd_min:" + date_start + ",cd_max:" + date_final),\
              "num": "100", 'q': (j + ' after:' + dt.strptime(date_start, '%m/%d/%Y').strftime('%Y-%m-%d') + ' before:' +  dt.strptime(date_final, '%m/%d/%Y').strftime('%Y-%m-%d')),\
              "hl": "en", "tz": "180", "start": str(i)}

    try:
      news_results = GoogleSearch(params).get_dict()['news_results']
      urls.extend([i['link'] for i in news_results])
    except:
      continue

df_news_urls = pd.DataFrame({'URL': urls})
output.eval_js('new Audio("https://upload.wikimedia.org/wikipedia/commons/0/05/Beep-09.ogg").play()')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

##### Save all urls:

In [30]:
df_urls_all = pd.concat([df_news_urls, df_news_urls], ignore_index=True)
df_urls_all = df_urls_all.drop_duplicates(subset='URL')

df_urls_all.to_excel('case_urls.xlsx', index=False)
files.download('case_urls.xlsx')

news_roots = list(frozenset([(tldextract.extract(i).domain  + '.' + tldextract.extract(i).suffix) for i in list(df_urls_all['URL'])]))

print('Number of urls:', len(list(df_urls_all['URL'])))
print('Number of sources:', len(news_roots), '\n')

Number of urls: 295
Number of sources: 131 



##### Add web content to signals:

In [42]:
df_content = df_urls_all.copy()
df_content['Content'] = parallel_text_func(list(df_content['URL']), parse_url, 10)

unwanted_phrases = ['Could not get content', 'JavaScript', 'Redirecting']
df_content = df_content[(df_content['Content'] != '') & (~df_content['Content'].str.contains('|'.join(unwanted_phrases), na=False))]

print('\nFraction of errors:', int(100*(1-len(df_content)/len(df_urls_all))), '%')
print('Number of parsed:', len(df_content))

output.eval_js('new Audio("https://upload.wikimedia.org/wikipedia/commons/0/05/Beep-09.ogg").play()')

100%|██████████| 295/295 [03:47<00:00,  1.30it/s]



Fraction of errors: 0 %
Number of parsed: 295


##### Filter content by publication date:

In [50]:
df_content["True Date"] = df_content["Content"].str.extract(r"#\s*(.+)$", expand=False)
df_content['True Date'] = pd.to_datetime(df_content['True Date'], format="%m/%d/%Y", errors='coerce')
df_content = df_content[df_content['True Date'].notna()]

df_content = df_content[(df_content['True Date'] >= dt.strptime(date_start, "%m/%d/%Y")) & (df_content['True Date'] < dt.strptime(date_final, "%m/%d/%Y"))]
del df_content['True Date']

print('\nFiltered articles:', len(df_content))
output.eval_js('new Audio("https://upload.wikimedia.org/wikipedia/commons/0/05/Beep-09.ogg").play()')


Filtered articles: 180


##### Extract cases:

In [68]:
df_final = df_content.copy()

df_final['Case'] = parallel_text_func(list(df_final['Content']), case_extraction, 10)
df_final = df_final[df_final['Case']!='NO']
df_final['Case_RU'] = parallel_text_func(list(df_final['Case']), case_russian, 10)

df_final = df_final.drop(columns=['Content'])
print('\nNumber of cases:', len(df_final))

output.eval_js('new Audio("https://upload.wikimedia.org/wikipedia/commons/0/05/Beep-09.ogg").play()')

100%|██████████| 117/117 [01:46<00:00,  1.10it/s]


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Number of cases: 117


##### Filter by relevance:

In [70]:
df_final['gpt_class'] = parallel_text_func(list(df_final['Case']), content_classification, 10)
df_final = df_final[df_final['gpt_class']==1]
df_final.drop(columns=['gpt_class'], inplace=True)

df_final.to_excel('Cases.xlsx', index=False)
files.download('Cases.xlsx')

print('\nNumber of relevant news:', len(df_final))
output.eval_js('new Audio("https://upload.wikimedia.org/wikipedia/commons/0/05/Beep-09.ogg").play()')

100%|██████████| 117/117 [00:10<00:00, 10.68it/s]


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Number of relevant news: 114
